# 02 — Retrieval, Generation, Citation, and Judge Metrics


## Mission

Diagnose where a RAG pipeline failed. First use exact information-retrieval metrics. Then evaluate answer behavior and calibrate semantic judges. Frameworks are adapters around this measurement design—not the definition of quality.


In [ ]:
from collections import Counter
import json, os, re, time
from pathlib import Path
import pandas as pd

from evaluation_contracts import *

pd.set_option("display.max_colwidth", 90)
corpus = load_corpus()
golden = load_cases()
print(f"Loaded {len(corpus)} corpus chunks and {len(golden)} golden cases.")


## 1. A transparent lexical retriever

This is intentionally simple: its scores are inspectable and its weaknesses create useful evaluation examples.


In [ ]:
TOKEN = re.compile(r"[a-z0-9]+")
def tokens(text): return set(TOKEN.findall(text.lower()))
def retrieve(query, k=5, include_stale=True):
    q = tokens(query)
    eligible = [c for c in corpus if include_stale or c.status == "current"]
    return [c.chunk_id for c in sorted(eligible, key=lambda c: (-len(q & tokens(c.text)), c.chunk_id))[:k]]

sample = next(c for c in golden if c.slice == "multi_evidence")
retrieved = retrieve(sample.query, k=5)
print(sample.query, retrieved, sep="\n")


## 2. Deterministic IR metrics first

Recall@k asks whether labelled relevant evidence appeared. Precision@k measures concentration. MRR rewards the first useful hit. nDCG uses graded relevance. Evidence completeness requires *all* indispensable chunks.


In [ ]:
relevance = {cid: (2 if cid in sample.required_evidence_ids else 1) for cid in sample.relevant_evidence_ids}
metric_row = {
    "Recall@5": recall_at_k(retrieved, set(sample.relevant_evidence_ids), 5),
    "Precision@5": precision_at_k(retrieved, set(sample.relevant_evidence_ids), 5),
    "MRR": reciprocal_rank(retrieved, set(sample.relevant_evidence_ids)),
    "nDCG@5": ndcg_at_k(retrieved, relevance, 5),
    "Evidence completeness": evidence_completeness(retrieved, set(sample.required_evidence_ids)),
}
display(pd.Series(metric_row).to_frame("score"))


In [ ]:
rows = []
for case in [c for c in golden if c.answerable][:20]:
    ids = retrieve(case.query, 5)
    rows.append({"case_id": case.case_id, "slice": case.slice,
        "recall@5": recall_at_k(ids, set(case.relevant_evidence_ids), 5),
        "mrr": reciprocal_rank(ids, set(case.relevant_evidence_ids)),
        "completeness": evidence_completeness(ids, set(case.required_evidence_ids))})
metric_df = pd.DataFrame(rows)
display(metric_df.groupby("slice")[["recall@5","mrr","completeness"]].mean().round(2))


## 3. Labelled context recall is not semantic sufficiency

With relevant chunk labels, recall is exact. With only a reference answer, a semantic sufficiency judge estimates whether context can support the answer. These are different measurements with different uncertainty.


In [ ]:
reference_only = {"reference": sample.reference_answer, "contexts": [next(c.text for c in corpus if c.chunk_id == x) for x in retrieved]}
print("Labelled context recall:", recall_at_k(retrieved, set(sample.relevant_evidence_ids), 5))
print("Semantic sufficiency input (requires calibrated human/model judgment):")
display(reference_only)


## 4. Generation behavior matrix

Correctness asks whether the answer is right. Faithfulness asks whether its claims follow from supplied context. Completeness and relevance remain separate.


In [ ]:
variants = pd.DataFrame([
 {"variant":"faithful_correct","answer":"Standard: 8 business hours; enterprise severity-one: 1 hour.","faithful":1,"correct":1,"complete":1,"relevant":1},
 {"variant":"faithful_incomplete","answer":"Standard support targets 8 business hours.","faithful":1,"correct":1,"complete":0,"relevant":1},
 {"variant":"faithful_stale_wrong","answer":"Leave carryover is 10 days until June 30.","faithful":1,"correct":0,"complete":1,"relevant":1},
 {"variant":"correct_unfaithful","answer":"The target is one hour, but no supplied source says so.","faithful":0,"correct":1,"complete":1,"relevant":1},
 {"variant":"unsupported_hallucination","answer":"Enterprise support responds in 15 minutes.","faithful":0,"correct":0,"complete":1,"relevant":1},
 {"variant":"irrelevant_but_faithful","answer":"MFA is required for workforce accounts.","faithful":1,"correct":0,"complete":0,"relevant":0},
])
display(variants)


## 5. Citation identity, correctness, and completeness

Validity: does the ID resolve? Correctness: does cited evidence support the associated claim? Completeness: are all material claims cited? Never collapse these into one boolean.


In [ ]:
def citation_scores(claims, corpus_ids):
    cited = [cid for claim in claims for cid in claim["citations"]]
    validity = sum(cid in corpus_ids for cid in cited) / len(cited) if cited else 0
    correctness = sum(claim["supported"] for claim in claims) / len(claims)
    completeness = sum(bool(claim["citations"]) for claim in claims) / len(claims)
    return {"validity": validity, "correctness": correctness, "completeness": completeness}

claims = [
 {"text":"Standard support targets eight business hours.","citations":["sla-standard#response"],"supported":True},
 {"text":"Enterprise severity-one targets one hour.","citations":[],"supported":True},
]
display(pd.Series(citation_scores(claims, {c.chunk_id for c in corpus})).to_frame("score"))


## 6. Current Ragas data contract—framework as adapter

Ragas exposes many metrics, not a fixed set of four. Its v0.4 collections API returns structured metric results. This offline cell builds its supported evaluation sample/dataset when installed; paid semantic metrics remain optional.


In [ ]:
try:
    from ragas import EvaluationDataset
    from ragas.dataset_schema import SingleTurnSample
    ragas_dataset = EvaluationDataset(samples=[SingleTurnSample(
        user_input=sample.query,
        response=sample.reference_answer,
        retrieved_contexts=reference_only["contexts"],
        reference=sample.reference_answer,
    )])
    print("Ragas dataset features:", ragas_dataset.features())
except Exception as exc:
    ragas_dataset = None
    print("Ragas adapter skipped; deterministic course metrics still ran:", type(exc).__name__, exc)


## 7. Structured semantic judge: real provider optional, frozen fallback required

Record the model, prompt, rubric, and temperature. The frozen outputs are reproducibility fixtures—not fabricated live results.


In [ ]:
JUDGE_CONFIG = {"model": os.getenv("JUDGE_MODEL", "frozen-small-v1"), "prompt_version":"correctness-v2", "rubric_version":"rag-correctness-2026-08", "temperature":0}

def optional_judge():
    if os.getenv("RUN_REAL_JUDGE") != "1": return None
    if not os.getenv("OPENAI_API_KEY"): raise RuntimeError("RUN_REAL_JUDGE=1 requires OPENAI_API_KEY")
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model=os.getenv("JUDGE_MODEL", "gpt-4o-mini"), temperature=0).with_structured_output(JudgeResult, method="json_schema")

print(JUDGE_CONFIG)
print("Live judge enabled:", optional_judge() is not None)


## 8. Calibrate against 20 human-labelled pairs

Agreement is measured, not assumed. Inspect a confusion table and Cohen's kappa; investigate disagreements by slice before trusting an aggregate.


In [ ]:
cal = pd.DataFrame(load_json("frozen_judge_calibration.json"))
comparisons = []
for column in ["frozen_judge_small_v1", "frozen_judge_strict_v2"]:
    comparisons.append({"judge":column, "accuracy":accuracy(cal.human_label.tolist(), cal[column].tolist()),
                        "kappa":cohen_kappa(cal.human_label.tolist(), cal[column].tolist())})
display(pd.DataFrame(comparisons))
display(pd.crosstab(cal.human_label, cal.frozen_judge_small_v1, rownames=["human"], colnames=["judge"]))
assert len(cal) == 20


## 9. Slice-level diagnosis and experiment design


In [ ]:
display(metric_df.groupby("slice").agg(cases=("case_id","count"), recall=("recall@5","mean"), completeness=("completeness","mean")).round(2))
print("Do not select a judge by size. Select a versioned configuration by measured agreement, cost, latency, stability, and slice behavior.")


## Exercises and production upgrades

1. Compare `k=3` and `k=8`; explain the precision/completeness trade-off.
2. Add graded relevance labels and inspect nDCG when required evidence is ranked late.
3. Run a real structured judge, save outputs separately, and compare them with the frozen fixtures.
4. Add Phoenix, LangSmith, MLflow, or DeepEval as an experiment store without changing metric definitions.

**Checkpoint:** exact labels support exact metrics; semantic properties need calibrated judges or human review.
